# Recalibración natural: theta_nv2 (cinco estados)

Se eliminan únicamente los estados de biomasa muerta y glicerol y sus parámetros de producción.
La mortalidad de la biomasa **viable**, Arrhenius, mantenimiento, inhibiciones y nutriciones se conservan.
Datos depurados LAB004–009; estados puntuados X, YAN, glucosa, fructosa y etanol, sin puntuar la entrada inicial.
La capa de CO₂ se conserva y se comprueba en el segundo notebook; no interviene en este objetivo químico.

Inicio exacto: **theta_full**, incluso su Y_E/F inicialmente no factible. Se usa SLSQP para entrar en la región
con ambos rendimientos ≤0.5114, seguido de mínimos cuadrados acotados. Sin multistart ni cambios de iniciales.
Los vectores antiguos se comparan sin corregir sus rendimientos.

Las tablas y figuras se conservan dentro del EXECUTED, sin exportar CSV o PNG auxiliares.

In [ ]:
from pathlib import Path
import sys, json, time, ast, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, least_squares, Bounds
from dataclasses import replace
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/sources.csv').exists() and (p/'src').exists())
sys.dont_write_bytecode=True
sys.path.insert(0,str(ROOT/'src'))
from wine_fermentation import model as base
from wine_fermentation.data import natural_batches
from wine_fermentation.historical import current_model as legacy
from wine_fermentation import co2 as gas

THETAS={name:base.normalize_parameters(json.loads((ROOT/f'configs/theta/{name}.json').read_text())) for name in ('theta_full','theta_new')}
raw_data,batches,temperature_profiles=natural_batches(ROOT)
CASES=('theta_full','theta_new','theta_nv2')
COLORS=dict(theta_full='#0072B2',theta_new='#D55E00',theta_nv2='#009E73')
plt.rcParams.update({'figure.dpi':115,'font.family':'DejaVu Sans'})
def save_json(path,value):path.write_text(json.dumps(value,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
display(pd.DataFrame([dict(lab=b.batch,X0=b.initials['X'],N0=b.initials['N'],G0=b.initials['G'],F0=b.initials['F'],E0=b.initials['E'],pulse_h=b.pulses['N'][0][0],dose_N=b.pulses['N'][0][1]) for b in batches]))

## Comprobación de la reducción

La proyección del RHS de siete estados debe coincidir con el nuevo RHS de cinco estados.
También se comparan integraciones de LAB004–009 con el mismo perfil térmico y nutrición.
Los cuerpos de las funciones históricas de producción, solubilidad, oxígeno y liberación de CO₂ deben ser idénticos.

In [ ]:
checks=[]
for name,theta in THETAS.items():
    old=json.loads((ROOT/f'configs/theta/{name}.json').read_text())
    for T in (12.,20.,28.,36.):
        for y in ([.3,.2,80.,80.,0.],[1.,.01,10.,20.,65.],[0.,0.,0.,0.,90.]):
            batch=replace(batches[0],time=np.array([0.,1.]),temperature_c=np.array([T,T]))
            y7=np.array([y[0],.2,y[1],y[2],y[3],y[4],5.])
            legacy_batch=legacy.BatchData(batch.medium,batch.batch,batch.time,batch.temperature_c,batch.pulses,batch.observations,batch.initials)
            a=np.asarray(legacy.rhs(0.,y7,old,legacy_batch))[[0,2,3,4,5]]
            error=float(np.max(abs(a-base.rhs(0.,y,theta,batch))))
            assert error<1e-12
    for batch in batches:
        old_batch=legacy.BatchData(batch.medium,batch.batch,batch.time,batch.temperature_c,batch.pulses,
            batch.observations,{**batch.initials,'Xd':0.,'Gly':0.})
        a=legacy.simulate(old_batch,old,integration_max_step_h=.5)
        assert a is not None
        b=base.integrate(batch,theta)
        delta=np.max(abs(a[list(base.STATE_NAMES)].to_numpy()-b.to_numpy()),axis=0)
        for state,error in zip(base.STATE_NAMES,delta):
            checks.append(dict(theta=name,lab=batch.batch,state=state,max_absolute_difference=float(error)))
        assert delta.max()<.002
oldtree=ast.parse((ROOT/'src/wine_fermentation/historical/current_co2.py').read_text())
newtree=ast.parse((ROOT/'src/wine_fermentation/co2.py').read_text())
for name in ('o2_saturation_mg_l','production','csat','release','finish'):
    a=next(n for n in oldtree.body if isinstance(n,ast.FunctionDef) and n.name==name)
    b=next(n for n in newtree.body if isinstance(n,ast.FunctionDef) and n.name==name)
    assert ast.dump(a)==ast.dump(b),name

print('RHS y funciones CO2 verificados. Máxima diferencia en integración:',pd.DataFrame(checks).max_absolute_difference.max())

## Parámetros y objetivo

Se liberan 9 parámetros físicos: tasas máximas, rendimiento de crecimiento sobre N, inhibiciones,
mortalidad y los dos rendimientos de etanol. Se fijan K_N0, K_G0, K_F0, Y_X/G, Y_X/F y m0 en theta_full.
**Fijar estas magnitudes físicas no equivale a fijar los antiguos cocientes s o q:** esta es una decisión explícita
para evitar mover simultáneamente tasas y todos sus rendimientos/constantes con seis fermentaciones.

Las cotas amplias son límites operativos del ajuste, no máximos teóricos universales.
El límite estequiométrico impuesto es 0.5114 g etanol/g azúcar para ambos rendimientos.
Cada residuo usa max(error absoluto histórico, error relativo histórico). X y YAN tienen error relativo 8%; G,F,E 2.5%.
Se incluyen todas las muestras finitas posteriores a t=0, sin crear observaciones de G+F adicionales.

In [ ]:
FREE=('mu0','Yxn','betaG0','betaF0','Yeg','Yef','Kig0','Kie0','Kd0')
LIMITS={'mu0':(.05,1.),'Yxn':(1.,100.),'betaG0':(.1,10.),'betaF0':(.1,10.),
    'Yeg':(.1,10.),'Yef':(.1,10.),'Kig0':(10.,1000.),'Kie0':(10.,1000.),'Kd0':(1e-4,.05)}
YMAX=.5114
INITIAL=THETAS['theta_full'].copy()
x0=np.log([INITIAL[n] for n in FREE]);lo=np.log([LIMITS[n][0] for n in FREE]);hi=np.log([LIMITS[n][1] for n in FREE])
yidx=np.array([FREE.index('Yeg'),FREE.index('Yef')])
assert np.all((x0>=lo)&(x0<=hi))
support=[]
for batch in batches:
    for state in base.STATE_NAMES:
        observed=batch.observations[state];mask=np.isfinite(observed)&(batch.time>0)
        support.append((batch,state,mask,observed[mask],base.sigma_for_state(state,observed[mask])))
NRES=sum(len(item[3]) for item in support)
def unpack(x):return {**INITIAL,**dict(zip(FREE,np.exp(x)))}
calls=0;last_x=None;last_r=None;rejected_trials=0
def residual(x,tight=False):
    global calls,last_x,last_r,rejected_trials
    if not tight and last_x is not None and np.array_equal(x,last_x):return last_r.copy()
    theta=unpack(x)
    try:
        predictions={b.batch:base.integrate(b,theta,rtol=2e-9 if tight else 2e-7,atol=2e-11 if tight else 2e-9,max_step=.5 if tight else .75) for b in batches}
    except RuntimeError:
        if tight:raise
        rejected_trials+=1
        return np.full(NRES,1000.+np.linalg.norm(x-x0))
    r=np.concatenate([(predictions[b.batch][s].to_numpy()[mask]-obs)/sigma for b,s,mask,obs,sigma in support])
    if not np.isfinite(r).all():raise ValueError('Nonfinite residual')
    if not tight:calls+=1;last_x=np.array(x).copy();last_r=r.copy()
    return r
def objective(x):return float(np.mean(residual(x)**2))
start=time.monotonic();initial_objective=objective(x0)

display(pd.DataFrame([dict(parameter=n,theta_full=INITIAL[n],calibrate=n in FREE,
    lower=LIMITS[n][0] if n in FREE else np.nan,upper=(YMAX if n in ('Yeg','Yef') else LIMITS[n][1]) if n in FREE else np.nan) for n in base.PARAMETERS]))
print('Residuales:',NRES,'objetivo inicial:',initial_objective,'segundos/evaluación:',time.monotonic()-start)

In [ ]:
trace=[]
def callback(x):
    row=dict(iteration=len(trace)+1,objective=objective(x),calls=calls,elapsed_s=time.monotonic()-start,
        Yeg=float(np.exp(x[FREE.index('Yeg')])),Yef=float(np.exp(x[FREE.index('Yef')])))
    trace.append(row)


    if len(trace)%10==0:print('SLSQP',row,flush=True)
constraint={'type':'ineq','fun':lambda x:np.log(YMAX)-x[yidx],
    'jac':lambda x:-np.eye(len(FREE))[yidx]}
stage1=minimize(objective,x0,method='SLSQP',bounds=Bounds(lo,hi),constraints=[constraint],callback=callback,
    options=dict(maxiter=160,ftol=1e-8,eps=2e-4,disp=True))
feasible=[np.array(stage1.x)] if np.all(stage1.x[yidx]<=np.log(YMAX)+1e-7) else []
assert feasible,'SLSQP no alcanzó una solución factible; no se publica theta_nv2'
upper=hi.copy();upper[yidx]=np.log(YMAX)
# Only numerical rounding of a constrained result is corrected here, never the theta_full start.
refinement_start=np.minimum(np.maximum(stage1.x,lo+1e-12),upper-1e-12)
stage2=least_squares(residual,refinement_start,bounds=(lo,upper),diff_step=2e-4,max_nfev=140,
    ftol=2e-7,xtol=2e-7,gtol=2e-5,verbose=1)
candidates=[('slsqp',stage1.x),('bounded_least_squares',stage2.x)]
selected,xfinal=min(candidates,key=lambda item:float(np.mean(residual(item[1],tight=True)**2)))
THETAS['theta_nv2']=unpack(xfinal)
assert all(0<THETAS['theta_nv2'][n]<=YMAX+1e-9 for n in ('Yeg','Yef'))
rough=residual(xfinal);accurate=residual(xfinal,tight=True)
numerical_delta=float(np.max(abs(rough-accurate)))
assert numerical_delta<.025,'Se requiere refinar integración antes de publicar'
save_json(ROOT/'configs/theta/theta_nv2.json',THETAS['theta_nv2'])
record=dict(initial_theta='theta_full',initial_exact=True,parameterization='physical',free=list(FREE),
    fixed={n:INITIAL[n] for n in base.PARAMETERS if n not in FREE},yield_maximum=YMAX,
    training_labs=[b.batch for b in batches],holdout_labs=['LAB019','LAB020','LAB021'],
    objective='mean squared standardized residual; X,N,G,F,E; t>0; no priors',
    slsqp=dict(success=bool(stage1.success),status=int(stage1.status),message=str(stage1.message),iterations=int(stage1.nit)),
    refinement=dict(success=bool(stage2.success),status=int(stage2.status),message=str(stage2.message),nfev=int(stage2.nfev),optimality=float(stage2.optimality)),
    selected=selected,initial_objective=initial_objective,final_objective=float(np.mean(accurate**2)),
    max_residual_change_tight_integration=numerical_delta,total_calls=calls,rejected_invalid_trials=rejected_trials,elapsed_s=time.monotonic()-start,
    interpretation='Local constrained fit, not a global optimum or proof of parameter identifiability')
save_json(ROOT/'configs/recalibration_nv2.json',record)
display(record)

## Comparación con theta_full y theta_new

Todos se evalúan con las mismas observaciones, iniciales, nutriciones, temperatura y modelo de cinco estados.
Se muestran RMSE físicos y RMSE ponderados; G+F se compara como diagnóstico, sin duplicarlo en el objetivo.
Un mejor objetivo conjunto puede coexistir con un peor ajuste en una fermentación o variable.

In [ ]:
rows=[];trajectories=[];weighted=[]
for name,theta in THETAS.items():
    all_res=[]
    for batch in batches:
        pred=base.integrate(batch,theta)
        output=pred.copy();output['GF']=output.G+output.F;output['time_h']=output.index;output['lab']=batch.batch;output['theta']=name
        trajectories.append(output.reset_index(drop=True))
        for state in (*base.STATE_NAMES,'GF'):
            observed=batch.observations[state] if state!='GF' else batch.observations['G']+batch.observations['F']
            mask=np.isfinite(observed)&(batch.time>0)
            error=output[state].to_numpy()[mask]-observed[mask]
            sigma=base.sigma_for_state(state,observed[mask]) if state!='GF' else np.sqrt(base.sigma_for_state('G',batch.observations['G'][mask])**2+base.sigma_for_state('F',batch.observations['F'][mask])**2)
            rows.append(dict(theta=name,lab=batch.batch,state=state,n=int(mask.sum()),rmse=float(np.sqrt(np.mean(error**2))),bias=float(np.mean(error)),weighted_rmse=float(np.sqrt(np.mean((error/sigma)**2)))))
            if state!='GF':all_res.extend(error/sigma)
    weighted.append(dict(theta=name,n=len(all_res),objective=float(np.mean(np.array(all_res)**2)),weighted_rmse=float(np.sqrt(np.mean(np.array(all_res)**2)))))
metrics=pd.DataFrame(rows);summary=pd.DataFrame(weighted)


parameters=pd.DataFrame(THETAS).rename_axis('parameter')
audit=pd.DataFrame([dict(theta=name,Yeg=t['Yeg'],Yef=t['Yef'],Yeg_feasible=t['Yeg']<=YMAX,Yef_feasible=t['Yef']<=YMAX,
    all_parameters_positive=all(v>0 for v in t.values()),fixed_parameters_preserved=all(t[n]==INITIAL[n] for n in base.PARAMETERS if n not in FREE) if name=='theta_nv2' else None) for name,t in THETAS.items()])

display(summary);display(audit);display(metrics.pivot(index=['lab','state'],columns='theta',values='rmse').round(4))

In [ ]:
LABELS={'X':'Biomasa viable [g/L]','N':'YAN [g N/L]','G':'Glucosa [g/L]','F':'Fructosa [g/L]','GF':'Glucosa + fructosa [g/L]','E':'Etanol [g/L]'}
for state in ('X','N','G','F','GF','E'):
    fig,axes=plt.subplots(2,3,figsize=(14,7),sharex=True)
    for ax,batch in zip(axes.flat,batches):
        grid=np.unique(np.r_[np.arange(0,batch.time[-1],.5),batch.time[-1],batch.time])
        for name,theta in THETAS.items():
            p=base.integrate(batch,theta,grid)
            ax.plot(grid,p.G+p.F if state=='GF' else p[state],color=COLORS[name],label=name,ls='--' if name=='theta_new' else '-')
        obs=batch.observations['G']+batch.observations['F'] if state=='GF' else batch.observations[state]
        mask=np.isfinite(obs)&(batch.time>0);ax.scatter(batch.time[mask],obs[mask],c='black',s=17,zorder=5)
        initial=batch.initials['G']+batch.initials['F'] if state=='GF' else batch.initials[state]
        ax.scatter([0],[initial],facecolor='white',edgecolor='black',s=30,zorder=6)
        ax.axvline(batch.pulses['N'][0][0],color='gray',alpha=.5,ls=':');ax.set_title(batch.batch);ax.grid(alpha=.3)
        ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel(LABELS[state])
    handles,labels=axes.flat[0].get_legend_handles_labels();fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,1.045),ncol=3)
    fig.suptitle(LABELS[state]+' · mosto natural LAB004–009',y=1.09);fig.tight_layout()
    plt.show()

## Alcance

La reducción conserva las trayectorias al mantener parámetros; las diferencias de theta_nv2 provienen del nuevo ajuste
de cinco variables y de las restricciones/selección explícita de parámetros. No se calibran iniciales, leyes térmicas
ni mantenimiento. Esta ejecución no demuestra identificabilidad ni resuelve por sí sola discrepancias de LAB006.
La evaluación posterior LAB019–021 y la comparación de CO₂ están en `02_co2_theta_nv2`.